# Exploring Emotional Profiles in 1.38M TMDB Movies

## Input

| Dataset | Description |
|---------|-------------|
| [TMDB Movie VAD + Emotion Intensity Scores](https://www.kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores) | 1.38M movies scored with NRC VAD + emotion intensity lexicons |

Mounted at: `/kaggle/input/datasets/bdelanghe/tmdb-movie-vad-emotion-scores/movie_vad_scores.csv`

## Output

This is an **analysis notebook** — no files are written. All outputs are inline visualizations and tables exploring the emotional profiles in the dataset.

---

This dataset combines the full TMDB movie catalog with affective scores from two NRC lexicons:

- **NRC VAD Lexicon** — valence (positive/negative), arousal (calm/excited), dominance (submissive/powerful)
- **NRC Emotion Intensity Lexicon** — continuous 0–1 scores for anger, anticipation, disgust, fear, joy, sadness, surprise, trust

Scores are derived from plot keywords, not reviews or text. See methodological notes at the end.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Input: TMDB Movie VAD + Emotion Intensity Scores
# https://www.kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores
INPUT_PATH = '/kaggle/input/datasets/bdelanghe/tmdb-movie-vad-emotion-scores/movie_vad_scores.csv'

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'Rows: {len(df):,}   Columns: {len(df.columns)}')
print(f'\nColumn groups:')
print('  Identity:  ', [c for c in df.columns[:3]])
print('  Sentiment: ', [c for c in df.columns[3:8]])
print('  Emotions:  ', [c for c in df.columns[8:15]])
print('  TMDB meta: ', [c for c in df.columns[15:20]], '...')

## Schema and coverage

In [ ]:
EMOTIONS = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']
VAD = ['valence', 'arousal', 'dominance']

print('Sentiment distribution:')
print(df['sentiment'].value_counts().to_string())
print()

scored = df[df['sentiment'] != 'unknown']
print(f'Movies with keyword coverage: {len(scored):,} ({100*len(scored)/len(df):.1f}%)')
print(f'\nEmotion score means (scored movies only):')
print(scored[EMOTIONS].mean().round(3).to_string())
print(f'\nVAD means:')
print(scored[VAD].mean().round(3).to_string())

## Top films by emotion

In [ ]:
# Movies with strongest emotion signal (min 5 keywords for reliability)
has_kw = df[df['keywords'].notna() & (df['keywords'].str.count(',') >= 4)]

print('Most joyful (by keyword associations):')
display(has_kw.nlargest(8, 'joy')[['title', 'genres', 'joy', 'sadness', 'sentiment']].reset_index(drop=True))

print('\nMost fear-associated:')
display(has_kw.nlargest(8, 'fear')[['title', 'genres', 'fear', 'anger', 'sentiment']].reset_index(drop=True))

print('\nMost negative valence:')
display(has_kw.nsmallest(8, 'valence')[['title', 'genres', 'valence', 'arousal', 'sentiment']].reset_index(drop=True))

## Genre-level emotional fingerprints

Genres are comma-separated — explode and aggregate to get the VAD + emotion profile per genre.

In [ ]:
# Explode genres, join scores, aggregate
genre_df = (
    df[df['genres'].notna() & (df['valence'].notna())]
    .assign(genre=df['genres'].str.split(', '))
    .explode('genre')
)

FEATURES = VAD + EMOTIONS
genre_means = (
    genre_df.groupby('genre')[FEATURES]
    .mean()
    .round(3)
)
# Filter to genres with enough films
genre_counts = genre_df.groupby('genre').size()
genre_means = genre_means[genre_counts >= 200].copy()
genre_means['n_movies'] = genre_counts[genre_means.index]

print('Genre emotional profiles (top genres by movie count):')
display(genre_means.sort_values('n_movies', ascending=False).head(15))

## Visualisation: VAD map of genres

Plot genres in valence × arousal space, sized by dominance.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

top_genres = genre_means.sort_values('n_movies', ascending=False).head(18)
colors = cm.tab20(np.linspace(0, 1, len(top_genres)))

for (genre, row), color in zip(top_genres.iterrows(), colors):
    size = 200 + row['dominance'] * 600
    ax.scatter(row['valence'], row['arousal'], s=size, color=color, alpha=0.85, zorder=3)
    ax.annotate(genre, (row['valence'], row['arousal']),
                fontsize=9, ha='center', va='bottom',
                xytext=(0, 7), textcoords='offset points')

ax.axvline(0.5, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axhline(0.5, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Valence  (0 = negative  →  1 = positive)', fontsize=11)
ax.set_ylabel('Arousal  (0 = calm  →  1 = excited)', fontsize=11)
ax.set_title('Genre Emotional Map — NRC VAD scores from TMDB keywords\n(dot size = dominance)', fontsize=12)
ax.set_xlim(0.3, 0.75)
ax.set_ylim(0.3, 0.65)
ax.text(0.32, 0.31, 'Low energy, negative', fontsize=8, color='gray', alpha=0.7)
ax.text(0.62, 0.62, 'High energy, positive', fontsize=8, color='gray', alpha=0.7)
plt.tight_layout()
plt.savefig('genre_vad_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved genre_vad_map.png')

## Caveats

1. **Scores are relative, not absolute.** Valence 0.65 means "more positive-associated than 0.55" — not "65% positive."
2. **Keyword associations, not audience emotion.** A horror film with keywords like *ghost, murder, fear* gets a high fear score because those words are fear-associated — not because audiences felt afraid.
3. **~75% of rows have no keywords** and are scored as `unknown`. The TMDB dataset is large and many entries are sparsely populated.
4. **Dominant sense bias.** Film-industry terms (shot, cut, cast) may carry non-cinematic scores from the lexicon's general-English training.
5. **Cultural perception.** The NRC lexicons were built via crowdsourcing and reflect majority annotator perception, not universal associations.

---

**Lexicon sources:** [saifmohammad.com/WebPages/lexicons.html](https://saifmohammad.com/WebPages/lexicons.html)  
**Pipeline source:** [github.com/bdelanghe/imdb-kaggle](https://github.com/bdelanghe/imdb-kaggle)  
**Reference:** Mohammad, S. M. (2020). Practical and Ethical Considerations in the Effective use of Emotion and Sentiment Lexicons.